# <b>Detect Color (with Image)</b>

In [ ]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

def display_image(image1, image2):
    """두 이미지를 가로로 결합하여 주피터에서 표시"""
    # 이미지를 가로로 결합
    combined_image = np.hstack((image1, image2))

    # 이미지를 JPEG 형식으로 인코딩
    _, jpeg = cv2.imencode('.jpg', combined_image)

    # 위젯을 업데이트하여 이미지 표시
    image_widget.value = jpeg.tobytes()

# 이미지 파일 경로
img_path = "RGB.png"

# 이미지 읽기
image = cv2.imread(img_path)

# 이미지 로드 확인
if image is None:
    raise ValueError(f"Error: Image not loaded from path: {img_path}. Please check the file path.")

# BGR을 HSV 컬러 스페이스로 변환
hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

# 흰색 영역 검출
lower_white = np.array([0, 0, 200], dtype=np.uint8)
upper_white = np.array([255, 30, 255], dtype=np.uint8)
white_mask = cv2.inRange(hsv, lower_white, upper_white)
white_result = cv2.bitwise_and(image, image, mask=white_mask)

# 이미지 디스플레이 위젯 생성
image_widget = widgets.Image(format='jpeg')
display(image_widget)

# 결과 이미지 표시
display_image(image, white_result)

# <b>Detect Color (with Camera)</b>

In [ ]:
from picamera2 import Picamera2, Preview
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

# 카메라 객체 생성
picam2 = Picamera2()
camera_config = picam2.create_preview_configuration(
    main={"size": (320, 180), "format": "BGR888"}
)
picam2.configure(camera_config)
picam2.start()

# 이미지를 표시할 위젯 생성
image_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='320px', height='180px'))
display(image_widget)

def update_image(frame1, frame2):
    """이미지 프레임을 수평으로 결합하여 위젯에 업데이트"""
    # 두 이미지를 수평으로 결합
    combined_frame = np.hstack((frame1, frame2))
    # 이미지를 JPEG 형식으로 인코딩
    _, jpeg = cv2.imencode('.jpg', combined_frame)
    # 위젯을 업데이트하여 이미지 표시
    image_widget.value = jpeg.tobytes()

try:
    while True:
        frame = picam2.capture_array()
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        lower_white = np.array([0, 0, 168], dtype=np.uint8)
        upper_white = np.array([172, 111, 255], dtype=np.uint8)
        white_mask = cv2.inRange(hsv, lower_white, upper_white)
        white_result = cv2.bitwise_and(frame, frame, mask=white_mask)

        # 영상 업데이트
        update_image(frame, white_result)

        # 약간의 대기 시간을 추가하여 프레임 간 간격을 제어
        time.sleep(0.2)  # 5 FPS로 설정

except KeyboardInterrupt:
    pass

finally:
    clear_output(wait=True)  # 이전 출력을 지우고 종료 메시지를 출력
    print("캡처 종료됨.")
